In [ ]:
import time
import pandas as pd
from sklearn.metrics import accuracy_score
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
results = []

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/MedAssist AI/clean_190k_dataset.csv')

In [ ]:
from sklearn.preprocessing import LabelEncoder
df_clean = df.copy()
# STEP 1: LABEL ENCODING
print("Translating diseases to numbers...")
encoder = LabelEncoder()
df_clean['target'] = encoder.fit_transform(df_clean['diseases'])
df_final = df_clean.drop(columns=['diseases'])

Translating diseases to numbers...


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("Isolating rare diseases to protect them from the split...")

# 1. Figure out which diseases are rare (1 row) and which are common (2+ rows)
class_counts = df_final['target'].value_counts()
rare_classes = class_counts[class_counts == 1].index
common_classes = class_counts[class_counts > 1].index

# 2. Split the dataset into two separate dataframes
df_rare = df_final[df_final['target'].isin(rare_classes)]
df_common = df_final[df_final['target'].isin(common_classes)]

# 3. Perform the Stratified Split ONLY on the common diseases
X_common = df_common.drop(columns=['target'])
y_common = df_common['target']

X_train_common, X_test, y_train_common, y_test = train_test_split(
    X_common, y_common, test_size=0.2, random_state=42, stratify=y_common
)

# 4. Manually force all the rare diseases directly into the Training Set
X_rare = df_rare.drop(columns=['target'])
y_rare = df_rare['target']

X_train = pd.concat([X_train_common, X_rare])
y_train = pd.concat([y_train_common, y_rare])

print("--- DATA PRESERVATION COMPLETE ---")
print(f"Total diseases preserved: {len(y_train.unique())}")
print(f"Training on {X_train.shape[0]} rows...")
print(f"Testing on {X_test.shape[0]} rows...")

Isolating rare diseases to protect them from the split...
--- DATA PRESERVATION COMPLETE ---
Total diseases preserved: 773
Training on 151726 rows...
Testing on 37921 rows...


In [ ]:
# Define the evaluation set for live tracking
eval_set = [(X_train, y_train), (X_test, y_test)]

In [ ]:
import re

# Instantly strip all illegal JSON characters from column names
X_train = X_train.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '_', x))
X_test = X_test.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '_', x))

# Re-define eval_set so it uses the newly cleaned X_test
eval_set = [(X_train, y_train), (X_test, y_test)]

print("--- Column Names Cleaned for LightGBM! ---")

--- Column Names Cleaned for LightGBM! ---


In [ ]:
lgb_model = lgb.LGBMClassifier(
    random_state=42,
    objective="multiclass",

    n_estimators=300,
    learning_rate=0.05,

    num_leaves=63,
    max_depth=12,

    min_child_samples=10,

    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,

    n_jobs=-1
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=eval_set,
    callbacks=[lgb.early_stopping(30, verbose=True)]
)

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

LGBMClassifier(bagging_fraction=0.8, bagging_freq=5, feature_fraction=0.8,
               learning_rate=0.05, min_child_samples=1, n_estimators=300,
               n_jobs=-1, num_leaves=127, objective='multiclass',
               random_state=42)

In [ ]:
t0 = time.time() # Start timer for prediction and evaluation
lgb_preds = lgb_model.predict(X_test)
acc_lgb = accuracy_score(y_test, lgb_preds) * 100 # Use lgb_preds directly
prec_lgb = precision_score(y_test, lgb_preds, average='weighted', zero_division=0) * 100
rec_lgb = recall_score(y_test, lgb_preds, average='weighted', zero_division=0) * 100
f1_lgb = f1_score(y_test, lgb_preds, average='weighted', zero_division=0) * 100
results.append({
    'Model': 'LightGBM',
    'Accuracy (%)': round(acc_lgb, 2),
    'Precision (%)': round(prec_lgb, 2),
    'Recall (%)': round(rec_lgb, 2),
    'F1-Score (%)': round(f1_lgb, 2),
    'Time (Mins)': round((time.time() - t0) / 60, 2)
})
print(pd.Series(lgb_preds).value_counts().head(10))

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
481    313
369    286
196    242
138    220
11     218
766    214
10     209
669    208
235    207
423    206
Name: count, dtype: int64


In [ ]:
print("\n" + "="*85)
print("                           FINAL MODEL COMPARISON (300 TREES)                           ")
print("="*85)
print(pd.DataFrame(results).to_string(index=False))


                           FINAL MODEL COMPARISON (300 TREES)                           
   Model  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)  Time (Mins)
LightGBM         60.77          75.18       60.77         65.65         0.05


In [ ]:
print(df_final["target"].value_counts().describe())
print(df_final["target"].value_counts().head(10))
print(df_final["target"].value_counts().tail(10))

count     773.000000
mean      245.338939
std       333.650389
min         1.000000
25%        10.000000
50%        77.000000
75%       315.000000
max      1219.000000
Name: count, dtype: float64
target
165    1219
481    1218
766    1218
138    1217
669    1216
145    1215
545    1215
235    1215
351    1215
742    1215
Name: count, dtype: int64
target
492    1
502    1
756    1
234    1
24     1
406    1
501    1
639    1
278    1
727    1
Name: count, dtype: int64


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score,
    top_k_accuracy_score,
    confusion_matrix,
    classification_report
)

# 1. Get standard predictions and probability predictions
# Standard predict gives the single most likely class (for F1 and Confusion Matrix)
y_pred = lgb_model.predict(X_test)
# predict_proba gives the percentage confidence for all 773 classes (required for Top-3)
y_prob = lgb_model.predict_proba(X_test)

# ==========================================
# 2. WEIGHTED & MACRO F1
# ==========================================
weighted_f1 = f1_score(y_test, y_pred, average='weighted')
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"✅ Weighted F1-Score: {weighted_f1:.4f}")
print(f"✅ Macro F1-Score: {macro_f1:.4f}")

# ==========================================
# 3. TOP-3 ACCURACY
# ==========================================
# This checks if the correct disease is in the model's top 3 highest-probability guesses
top3_acc = top_k_accuracy_score(y_test, y_prob, k=3, labels=encoder.classes_)
print(f"✅ Top-3 Accuracy: {top3_acc:.4f}\n")

# ==========================================
# 4. CONFUSION MATRIX
# ==========================================
cm = confusion_matrix(y_test, y_pred)
print(f"✅ Confusion Matrix generated. Shape: {cm.shape}")
# Note: Printing a 773x773 grid will crash the output window.
# It is stored in the 'cm' variable if you need to extract specific row data later.

# ==========================================
# 5. PER-CLASS METRICS: COMMON VS. RARE
# ==========================================
# Generate the full report as a dictionary so we can filter it mathematically
report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()

# Drop the summary rows at the bottom so we only have the actual disease classes
class_metrics = report_df.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')

# Define what counts as "Rare" vs "Common" using the 'support' column
# (Support = the number of patient records that exist in the test set for that disease)
rare_diseases = class_metrics[class_metrics['support'] <= 3]
common_diseases = class_metrics[class_metrics['support'] >= 30]

print("--- 🔬 METRICS FOR RARE DISEASES (3 or fewer test samples) ---")
# Displaying the average performance for the rare subset
print(rare_diseases[['precision', 'recall', 'f1-score']].mean())

print("\n--- 🔬 METRICS FOR COMMON DISEASES (30 or more test samples) ---")
# Displaying the average performance for the highly recurrent subset
print(common_diseases[['precision', 'recall', 'f1-score']].mean())

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
✅ Weighted F1-Score: 0.6565
✅ Macro F1-Score: 0.4059


ValueError: 'y_true' contains labels not in parameter 'labels'.